# Final Project: all 25 exercises completed

In [1]:
import os, json, copy, io, zipfile, shutil, random, warnings, requests
import numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torchvision.models as models
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
warnings.filterwarnings("ignore"); torch.manual_seed(0)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
u = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ExisQFol3hUHktTjm6a51w/final-project-stop-signs-1-2025-04-25-t-06-47-41-058-z.zip"
zipfile.ZipFile(io.BytesIO(requests.get(u).content)).extractall("nsd")
d = "nsd/final-project-stop-signs-1-2025-04-25-t-06-47-41-058-z"
ann = json.load(open(d + "/_annotations.json"))["annotations"]
lab = {}
for f, e in ann.items(): lab.setdefault(e[0]["label"], []).append(f)
for l, fs in lab.items():
    random.shuffle(fs); k = int(len(fs) * 0.9)
    for s, part in (("train", fs[:k]), ("val", fs[k:])):
        os.makedirs(f"dataset/{s}/{l}", exist_ok=True)
        for f in part: shutil.copy2(f"{d}/{f}", f"dataset/{s}/{l}/{f}")
print({l: len(v) for l, v in lab.items()})

{'not_stop': 100, 'stop': 97}


In [2]:
# Exercise 1: transformations
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
print("Exercise 1 completed")
# Exercise 2: datasets
train_dataset = ImageFolder(root="dataset/train", transform=transform)
val_dataset = ImageFolder(root="dataset/val", transform=transform)
print("Exercise 2 completed:", len(train_dataset), len(val_dataset))
# Exercise 3: visualize 3 validation images
i = 0
for x, y in val_dataset:
    plt.imshow(np.clip(x.permute(1, 2, 0).numpy() * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406], 0, 1)); plt.title(f"y = {y}"); plt.show()
    i += 1
    if i == 3: break
print("Exercise 3 completed: showed", i, "images")

Exercise 1 completed
Exercise 2 completed: 177 20


<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

Exercise 3 completed: showed 3 images


**Exercise 4 answer:** 1) Normalization matches the ImageNet statistics used in pretraining. 2) Resizing gives equal shapes for batching. 3) Varying sizes cannot be batched.

In [3]:
# Exercise 5: epochs and batch size
n_epochs = 10; batch_size = 32
print("Exercise 5 completed:", n_epochs, batch_size)
# Exercise 6: learning rate and momentum
learning_rate = 0.001; momentum = 0.9
print("Exercise 6 completed:", learning_rate, momentum)
# Exercise 7: scheduler settings
lr_scheduler = True; base_lr = 0.001; max_lr = 0.01
print("Exercise 7 completed:", lr_scheduler, base_lr, max_lr)

Exercise 5 completed: 10 32
Exercise 6 completed: 0.001 0.9
Exercise 7 completed: True 0.001 0.01


**Exercise 8 answer:** 1) Epochs: too few underfit, too many overfit. 2) Batch size affects stability, speed, memory. 3) A scheduler improves convergence.

In [4]:
# Exercise 9: pretrained model
model = models.resnet18(pretrained=True)
print("Exercise 9 completed")
# Exercise 10: freeze layers
for param in model.parameters(): param.requires_grad = False
print("Exercise 10 completed")
# Exercise 11: number of classes
n_classes = len(train_dataset.classes)
print("Exercise 11 completed:", n_classes)
# Exercise 12: new output layer
model.fc = nn.Linear(512, n_classes); model.to(device)
print("Exercise 12 completed")
# Exercise 13: loss
criterion = nn.CrossEntropyLoss()
print("Exercise 13 completed")
# Exercise 14: data loaders
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = torch.utils.data.DataLoader(dataset=val_dataset, batch_size=1)
print("Exercise 14 completed:", len(train_loader), len(validation_loader))
# Exercise 15: optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum)
print("Exercise 15 completed")
# Exercise 16: scheduler
if lr_scheduler:
    scheduler = torch.optim.lr_scheduler.CyclicLR(optimizer, base_lr=base_lr, max_lr=max_lr, step_size_up=5, mode="triangular2")
print("Exercise 16 completed")

Exercise 9 completed
Exercise 10 completed
Exercise 11 completed: 2
Exercise 12 completed
Exercise 13 completed
Exercise 14 completed: 6 20
Exercise 15 completed
Exercise 16 completed


In [5]:
# Exercise 17: training loop
def train_model(model, train_loader, validation_loader, criterion, optimizer, n_epochs, print_=True):
    loss_list, accuracy_list = [], []
    n_test = len(val_dataset); accuracy_best = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    for epoch in range(n_epochs):
        loss_sublist = []
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            model.train()
            loss = criterion(model(x), y)
            loss_sublist.append(loss.item())
            loss.backward(); optimizer.step(); optimizer.zero_grad()
        scheduler.step()
        loss_list.append(np.mean(loss_sublist))
        correct = 0; model.eval()
        with torch.no_grad():
            for xt, yt in validation_loader:
                _, yhat = torch.max(model(xt.to(device)).data, 1)
                correct += (yhat == yt.to(device)).sum().item()
        accuracy = correct / n_test; accuracy_list.append(accuracy)
        if accuracy > accuracy_best:
            accuracy_best = accuracy; best_model_wts = copy.deepcopy(model.state_dict())
        if print_: print(f"Epoch {epoch+1}: acc {accuracy:.2f}")
    model.load_state_dict(best_model_wts)
    return accuracy_list, loss_list, model
print("Exercise 17 completed")
accuracy_list, loss_list, model = train_model(model, train_loader, validation_loader, criterion, optimizer, n_epochs)
torch.save(model.state_dict(), "model.pt")

Exercise 17 completed


Epoch 1: acc 0.65


Epoch 2: acc 0.90


Epoch 3: acc 0.90


Epoch 4: acc 0.90


...

Epoch 6: acc 0.95


Epoch 7: acc 0.90


Epoch 8: acc 0.90


Epoch 9: acc 0.90


Epoch 10: acc 0.95


**Exercise 18 answer:** We would keep the last, possibly overfit, weights instead of the best ones.

In [6]:
# Exercise 19: plot loss and accuracy
fig, ax1 = plt.subplots(); ax1.plot(loss_list, color="tab:red"); ax1.set_xlabel("Iteration"); ax1.set_ylabel("Total Loss")
ax2 = ax1.twinx(); ax2.plot(accuracy_list, color="tab:blue"); ax2.set_ylabel("Accuracy"); plt.show()
print("Exercise 19 completed")

<Figure size 640x480 with 2 Axes>

Exercise 19 completed


**Exercise 20 answer:** 1. B 2. B

In [7]:
for p, f in [("6GVQqlNsZ83-me4L9DzAIg/not-stop-1.jpeg", "not-stop-1.jpeg")]:
    open(f, "wb").write(requests.get("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/" + p).content)
# Exercise 21: load trained model
class_names = ['not_stop', 'stop']
model = models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load("model.pt", map_location=torch.device('cpu')))
model.eval()
print("Exercise 21 completed")
# Exercise 22: preprocessing
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
print("Exercise 22 completed")
# Exercise 23: load and convert image
image = Image.open("not-stop-1.jpeg").convert("RGB")
input_tensor = transform(image).unsqueeze(0)
print("Exercise 23 completed:", tuple(input_tensor.shape))

Exercise 21 completed
Exercise 22 completed
Exercise 23 completed: (1, 3, 224, 224)


**Exercise 24 answer:** C. To add a batch dimension for the model.

In [8]:
# Exercise 25: prediction
with torch.no_grad():
    predicted_class = torch.argmax(model(input_tensor), dim=1).item()
print(f"Exercise 25 completed: the image was classified as {class_names[predicted_class]}")
plt.imshow(image); plt.title(f"Predicted: {class_names[predicted_class]}"); plt.axis("off"); plt.show()

Exercise 25 completed: the image was classified as not_stop


<Figure size 640x480 with 1 Axes>